# Building and Running Models

This notebook shows how to create neural network layers, compose them into
a model, run a forward pass, and train.

## Layer Types

idris-ml has a rich layer system. Let's explore what's available.

In [ ]:
:t linearLayer

In [ ]:
:t rnnLayer

In [ ]:
:t lstmLayer

In [ ]:
:t softmaxLayer

## Creating a Model

Layers are composed with `~>` and terminated with `OutputLayer`.
`autoName` assigns parameter names for gradient tracking.

In [ ]:
:exec do { ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (ll ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

## Forward Pass

Create an input tensor and run it through the model.
`forwardVarTensor` returns `(updatedModel, outputTensor)`.

In [ ]:
:exec do { ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (OutputLayer ll));
  buf <- pure (prim__setDouble (prim__setDouble (prim__allocDoubles 2) 0 1.0) 1 2.0);
  inT <- pure (prim__createState1d 2 buf);
  pair <- pure (forwardVarTensor model inT);
  outT <- pure (snd pair);
  putStrLn ("output[0] = " ++ show (prim__item1d outT 0));
  putStrLn ("output[1] = " ++ show (prim__item1d outT 1));
  putStrLn ("output[2] = " ++ show (prim__item1d outT 2));
  putStrLn ("sum = " ++ show (prim__item (prim__sum outT))) }

The output sums to 1.0 — the softmax output layer produces a probability distribution.

## Deeper Networks

Chain multiple layers with `~>`. Type-level dimension checking ensures
adjacent layers are compatible.

In [ ]:
:exec do { l1 <- linearLayer {i=4, o=8};
  l2 <- linearLayer {i=8, o=3};
  model <- pure (autoName (l1 ~> reluLayer ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

## Type Safety

Mismatched dimensions are caught at compile time. This won't type-check:

In [ ]:
:exec do { l1 <- linearLayer {i=4, o=8};
  l2 <- linearLayer {i=5, o=3};
  model <- pure (autoName (l1 ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn "should not reach here" }

The error shows a type mismatch between output dimension 8 and input dimension 5.
This is the key advantage of idris-ml: shape errors are caught before any code runs.

## Optimizers

idris-ml provides SGD, Adam, AdamW, and RMSprop. These are C-level native optimizers.

In [ ]:
:t nativeSgd

In [ ]:
:t nativeAdam

## Training

For full training loops, the compiled examples are more practical.
Run them from the terminal:

```bash
make example-supervised   # Linear classification (simplest)
make example-rnn          # RNN pattern prediction
make example-lstm         # LSTM sequence learning
make example-mnist        # CNN on MNIST
make example-gpt          # Character-level language model
make example-reinforce    # REINFORCE on CartPole
```

All examples accept `--epochs`, `--lr`, and `--seed` flags.

## Exploring the API

Use `:browse` to discover what's available in each module.

In [ ]:
:browse Optimizer

In [ ]:
:browse Layer.Conv

In [ ]:
:browse Schedule